In [1]:
"""
<텍스트 분할(Text Split)하는 이유>
 * 생성형 AI 언어모델은 입력 길이에 제한이 있음
   - 긴 문서를 적절한 크기로 분할하여 언어 모델에 적용하는 것이 매우 중요함


 * PDF 파일을 로드하여 사용하는 경우에는 적절한 크기로 분할하여 사용해야함
   - 적절한 크기로 분할하는 라이브러리를 랭체인에서 제공하고 있음
   - 이때 분할하는 라이브러리를 "텍스트 분할기(Text Splitter)"라고 칭합니다.


<텍스트 분할기(Text Splitter) 종류>
 1. 재귀적 문자 텍스트 분할기(RecursiveCharacterTextSplitter)
    : 긴 텍스트를 사용자가 지정한 최대 길이 또는 구분자를 기준으로, 반복적으로 분할함
    : 랭체인에서 가장 널리 사용되고 있음


 2. 의미적 텍스트 청크(문장) 분할기(SemanticChunker)
   *** CPU만 사용할 경우
       ===> 분할 시간이 매우 오래 걸림
       ===> 시스템 자원(GPU)을 고려하여 실행해야함


    : 긴 텍스트 내에서 의미적 유사성을 고려하여, 그룹으로 묶어서 문장(청크)으로 분류함
    : 긴 텍스트를 문장 단위로 분할한 후
        -> 서로 유사한 의미를 가진 문장들을 그룹화하여
        -> 각 그룹을 하나씩의 청크로 구성함(1개 그룹이 1개 청크가 됨)
    : 청크 내에서 앞/뒤 문맥이 잘 연결된 상태로 분할되어,
        -> 텍스트의 의미를 보존하면서도, 적절한 크기의 청크를 생성할 수 있음
        -> 모델의 응답 정확도가 크게 향상될 수 있음
    : 자연어 처리나 정보 검색에 연계하여 사용할 경우
        -> 더욱 정확한 결과를 얻을 수 있음


    <라이브러리 설치 필요>    
    - 의미 단위 분할기 라이브러리 설치
      >uv pip install langchain_experimental
"""


'\n<텍스트 분할(Text Split)하는 이유>\n * 생성형 AI 언어모델은 입력 길이에 제한이 있음\n   - 긴 문서를 적절한 크기로 분할하여 언어 모델에 적용하는 것이 매우 중요함\n\n\n * PDF 파일을 로드하여 사용하는 경우에는 적절한 크기로 분할하여 사용해야함\n   - 적절한 크기로 분할하는 라이브러리를 랭체인에서 제공하고 있음\n   - 이때 분할하는 라이브러리를 "텍스트 분할기(Text Splitter)"라고 칭합니다.\n\n\n<텍스트 분할기(Text Splitter) 종류>\n 1. 재귀적 문자 텍스트 분할기(RecursiveCharacterTextSplitter)\n    : 긴 텍스트를 사용자가 지정한 최대 길이 또는 구분자를 기준으로, 반복적으로 분할함\n    : 랭체인에서 가장 널리 사용되고 있음\n\n\n 2. 의미적 텍스트 청크(문장) 분할기(SemanticChunker)\n   *** CPU만 사용할 경우\n       ===> 분할 시간이 매우 오래 걸림\n       ===> 시스템 자원(GPU)을 고려하여 실행해야함\n\n\n    : 긴 텍스트 내에서 의미적 유사성을 고려하여, 그룹으로 묶어서 문장(청크)으로 분류함\n    : 긴 텍스트를 문장 단위로 분할한 후\n        -> 서로 유사한 의미를 가진 문장들을 그룹화하여\n        -> 각 그룹을 하나씩의 청크로 구성함(1개 그룹이 1개 청크가 됨)\n    : 청크 내에서 앞/뒤 문맥이 잘 연결된 상태로 분할되어,\n        -> 텍스트의 의미를 보존하면서도, 적절한 크기의 청크를 생성할 수 있음\n        -> 모델의 응답 정확도가 크게 향상될 수 있음\n    : 자연어 처리나 정보 검색에 연계하여 사용할 경우\n        -> 더욱 정확한 결과를 얻을 수 있음\n\n\n    <라이브러리 설치 필요>    \n    - 의미 단위 분할기 라이브러리 설치\n      >uv pip install langchain_experime

## 1. 재귀적 문자 텍스트 분할기(RecursiveCharacterTextSplitter)

In [2]:
### 긴 텍스트 문장을 사용자(개발자, 분석가..)가 지정한 최대 길이 또는 구분자를 기준으로, 반복적 분할 진행

#### 라이브러리 정의하기

In [4]:
### PDF 파일 읽어들이는 로더 클래스 정의(로더 여러개- 3가지 중 한가지 사용하면 됨)
from langchain_community.document_loaders import PyPDFLoader

### 재귀적 문자 텍스트 분할기 클래스 정의
from langchain_text_splitters import RecursiveCharacterTextSplitter

### PDF 파일 로드하기

In [12]:
### PDF 파일 읽어들인 후 pages 변수에 저장해 주세요.
# - 청크(문서)의 갯수 확인

### 파일 경로 변수 지정
file_path = "./data/2024_KB_부동산_보고서_최종.pdf"

### 파일 로더 클래스 생성
loader = PyPDFLoader(file_path)

### 파일 불러오고 변수에 저장
pages = loader.load()

### 청크(문서)의 갯수 확인
print(f"청크의 갯수 : {len(pages)}")
pages

청크의 갯수 : 84


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주

### 텍스트 분할기 클래스 생성하기

In [13]:
### 텍스트 분할기 클래스(전처리 클래스)
text_splitter = RecursiveCharacterTextSplitter(
    ### 텍스트를 일정 길이(청크 크기)로 구분하기 위한 기준 정의
    # - 문장 내에서 문자 500자를 기준으로 문장(청크)나누기
    # - 1청크(문장) = 500문자를 의미함
    chunk_size = 500,

    # - 문맥의 연속성을 유지하기 위하여 일정 문자를 이전과 이후 문장(청크)에
    #  -> 중복 되도록 처리하기 위한 문자의 길이 정의
    #  -> 이전 및 이후 문장(청크)에서 문자길이 50개씩은 중복해서 사용
    #     (중복이 안 일어 나는 경우도 있음)
    chunk_overlap = 50,

    # * 분할 기호 정의 : 기본(default)값은 ["\n\n", "\n", " ", ""], 생략가능
    #   : 기본값에 있는 리스트의 순서는 분할 우선순위를 의미함
    #   : 분할 시에 분할 기호의 우선순위를 기준으로 분할 진행 함
    #
    # * ["\n\n", "\n", " ", ""]
    #   : 텍스트를 무작정 자르지 않고, 의미가 연결된 문맥을 최대한 유지하기 위해
    #     이 리스트에 적힌 "순서대로 텍스트 분할"을 시도
    #   - 1순위 "\n\n" (두 줄 바꿈 / 문단 기준)
    #       : 가장 먼저 텍스트를 문단 단위로 분리
    #       : 문단은 글의 의미가 나뉘는 가장 큰 단위이므로,
    #         -> 문단끼리 묶어서 chunk_size(500자)를 맞추려고 시도함
    #   - 2순위 "\n" (한 줄 바꿈 / 문장 기준)
    #       : 문단 단위로 잘랐음에도 500자가 넘는 덩어리가 있다면,
    #       : 그 덩어리 안에서 줄바꿈 기호를 기준으로 다시 분리
    #   - 3순위 " " (띄어쓰기 / 단어 기준): 줄바꿈으로 잘랐는데도 여전히 500자가 넘는다면,
    #       :어쩔 수 없이 띄어쓰기를 기준으로 단어 단위로 분리
    #   - 4순위 "" (문자 기준): 띄어쓰기 조차 없는 아주 긴 텍스트(예: 500자가 넘는 URL이나 연속된 문자열)일 경우,
    #       : 최종적으로 글자 단위로 무조건 잘라내어 500자 크기를 맞춥니다.
    separators = ["\n\n", "\n", " ", ""] #디폴트값
)

text_splitter

### 텍스트 분할기 클래스를 이용하여 -> 분할하기

In [15]:
### split_documents : 텍스트 분할 함수(분할 기호를 기준으로 청크 사이즈 만큼 자르기 진행)
texts = text_splitter.split_documents(pages)

print(f"청크 갯수 : {len(texts)}")
texts

청크 갯수 : 218


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주요 변수들의 상황에 따라 \n소폭 반등 혹은 하락폭 확대 등의 방향이 정해질 것으로 보인다. \n2024년 주택시장의 주요 변수는 공급과 금리다. 급격하게 위축된 주택 공급이 단기간에 증가하기는 \n쉽지 않으나 정부의 공급 시그널이 지속된다면 일정 부분 해소가 될 가능성이 있다. 무엇보다 금리가 주요'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meetin

In [17]:
### 0번째 청크(문서) 본문 내용 출력...
print(f"본문 내용 : {texts[0].page_content}")

본문 내용 : 1 
2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 
 
 
 
Executive Summary 1 
 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 
주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 
가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 
시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 
부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주요 변수들의 상황에 따라 
소폭 반등 혹은 하락폭 확대 등의 방향이 정해질 것으로 보인다. 
2024년 주택시장의 주요 변수는 공급과 금리다. 급격하게 위축된 주택 공급이 단기간에 증가하기는 
쉽지 않으나 정부의 공급 시그널이 지속된다면 일정 부분 해소가 될 가능성이 있다. 무엇보다 금리가 주요


In [18]:
### 1번째 청크(문서) 본문 내용 출력...
print(f"본문 내용 : {texts[1].page_content}")

본문 내용 : 변수가 될 것이다. 기준금리 인하 시기와 인하 폭에 따라 주택 수요는 영향을 받을 수밖에 없기 때문이다. 
한편, 수요 위축으로 거래가 급감한 상황에서 실수요자 금융 지원, 관련 규제 완화 등 수요 회복을 위한 
정부 정책도 중요한 변수가 될 전망이다. 
 
 7대 이슈를 통해 바라보는 2024년 주택시장 
1 역대 최저 수준이 지속되고 있는 주택 거래 
 
주택 매매 거래는 2022년에 이어 2년 연속 침체. 총선 이후 정책 불확실성 해소와 금리 인하로 인한 회
복 가능성이 있으나 일부 지역 수요 쏠림 현상과 금리 인하 속도가 더딜 경우 회복세는 제한적일 전망 
2 주택공급 급격한 감소로 인한 공급 부족 가능성 
 
분양물량과 함께 장기적인 주택 공급 기반인 인허가물량까지 급감. 청약 수요 위축으로 분양 지연이 장기
화될 가능성이 높은 가운데 정부의 공급 정책 구체화가 매우 중요 
3 노후계획도시 특별법과 재건축 시장 영향


In [19]:
### 2번째 청크(문서) 본문 내용 출력...
print(f"본문 내용 : {texts[2].page_content}")

본문 내용 : 3 노후계획도시 특별법과 재건축 시장 영향 
 
2023년 말 국회를 통과한 「노후계획도시 특별법」 시행을 앞두고 당초 51곳이었던 대상 지역이 108곳으
로 확대. 단기적 효과는 제한적이나 사업진행이 구체화되면 시장 영향도 커질 것  
4 전세 수요 아파트 집중, 입주물량 부족으로 가격 상승 가능성 확대 
 
비아파트 전세 사기와 보증금 미반환 이슈 등으로 아파트로 전세 수요가 집중되는 가운데 수도권을 중심
으로 아파트 입주물량이 크게 감소함에 따라 다시 전세가격 상승세가 확대될 가능성이 존재 
5 주택 경기에 최대 화두로 부각되는 금리 인하 가능성 
 
금리 인하에 따른 매수 심리 회복에 대한 기대감이 높아지고 있지만 가계 부채 관리 강화와 경기 불확실
성 확대 등이 존재하는 시장에서 금리 인하 시기와 인하폭이 관건 
6 주택경기 위축에도 늘어나는 주택담보대출 
 
주택담보대출 증가세는 다소 둔화될 것으로 전망되나 가계부채에 대한 우려가 지속될 경우 매수세 회복을


## 2. 의미적 텍스트 청크(문장) 분할기(SemanticChunker)

In [ ]:
"""
<의미 단위 분할 과정>
 1. 문장을 임베딩(압축하는 것) 벡터로 변환하여 각 문장의 의미를 숫자로 변환
    - LLM 모델 중에 "실제 사용하는 모델에서 제공해 주는 임베딩 클래스"를 사용해야함(설치 필요)
    - 임베딩 처리로 인해 텍스트 분할 실행 시간이 다소 오래 걸림


 2. 인접한 문장 쌍 사이의 코사인 거리를 계산하여,
    - 문장 간의 의미적 차이가 어느 정도인지 파악


 3. 분할 시점(지점) 결정
    - 문장을 의미 단위로 분할할 시점(지점)을 결정하는 3가지 방식 중 한가지를 사용함


    * 분할 시점(지점) 결정 방식
      - 백분위수 방식(percentile)
        : 주로 사용되는 분할 방식
        : 문장 간 의미적 차이의 기준을 퍼센트(%)값으로 지정하여 분할함
        : 텍스트 내에서 문맥의 의미가 크게 달라지는 지점을 선택하여 청크를 나누게됨
        : 주로 95%를 기준으로함


      - 표준편차 방식(standard_deviation)
        : 문장 간의 의미적 차이가 표준편차 범위를 초과하는 시점(지점)을 기준으로 청크를 나눔
        : 문장 간의 의미 변화가 "평균"에서 얼마나 벗어나는지에 따라 분할 시점(지점)을 설정함
        : 분할 기준값 = 평균 코사인 거리 + (계수 x 표준편차)
        : 계수값이 작을 수록 -> 분할 시점(지점)이 많아지게 되며 -> 청크의 수가 커짐
        : 계수값은 주로 3을 사용함


      - 사분위수 방식(interquartile)
        : 텍스트 내에 중간(50% 데이터 분포) 정도의 의미적 변화를 감지하여 -> 분할 시점(지점)을 설정함
        : 분할 기준값 = Q3(3사분위값) + (계수 x IQR(사분위 범위:Q3-Q1))
        : 계수값이 작을 수록 -> 분할 시점(지점)이 많아지게 되며 -> 청크의 수가 커짐
        : 계수값은 주로 1.5를 사용함




 - 임베딩 전용 Ollama 모델 다운로드하기
   : 다운로드가 완료된 후(success 메시지 확인) 주피터 노트북(또는 파이썬 스크립트)에서
      -> 해당 코드를 다시 실행하면 정상적으로 문장 분할이 작동됨
   >ollama pull nomic-embed-text
"""


#### 라이브러리 정의하기

In [20]:
### PDF 문서 로더
from langchain_community.document_loaders import PyPDFLoader

### 임베딩(문자를 숫자로 변환)하는 모델
from langchain_ollama.embeddings import OllamaEmbeddings

### 의미 단위 분할기
from langchain_experimental.text_splitter import SemanticChunker

### 분할방식(1) : 백분위수 방식(percentile)

In [24]:
### pages 변수명으로 pdf 파일 새로 읽어들이기
loader = PyPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")

pages = loader.load()
print(f"청크(문서) 갯수 :{len(pages)}")
pages

청크(문서) 갯수 :84


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주

### 의미적 텍스트 분할기 클래스(SemanticChunker) 생성하기

In [26]:
### 의미적 텍스트 분할기 클래스 생성시에는 "분할 방법"과 "분할 기준"을 정의해야함
text_splitter = SemanticChunker(
    ### 문자를 숫자로 변환하는 임베딩 올라마 모델 정의
    embeddings = OllamaEmbeddings(model="nomic-embed-text"),

    ### 분할 방법 정의 : 백분위수 방법을 사용하겠다는 정의
    breakpoint_threshold_type = "percentile",

    ### 분할 기준 정의 (백분위 분할에서는 주로 95%를 사용)
    breakpoint_threshold_amount = 95
)

text_splitter

### 분할기를 이용하여 분할하기

In [28]:
### 분할기 함수를 사용
texts = text_splitter.split_documents(pages)

print(f"청크(문서)의 갯수 : {len(texts)}")
texts

청크(문서)의 갯수 : 164


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title'

### 분할방법(2) : 표준편차 방식(standard deviation)으로 분할하기

In [29]:
### 의미적 텍스트 분할기 클래스 생성시에는 "분할 방법"과 "분할 기준"을 정의해야함
text_splitter = SemanticChunker(
    ### 문자를 숫자로 변환하는 임베딩 올라마 모델 정의
    embeddings = OllamaEmbeddings(model="nomic-embed-text"),

    ### 분할 방법 정의 : 표준편차 방법을 사용하겠다는 정의
    breakpoint_threshold_type = "standard_deviation",

    ### 분할 기준 정의 (표준편차 분할에서는 주로 3을 사용)
    breakpoint_threshold_amount = 3
)

text_splitter

### 표준편차 방식으로 분할하기

In [30]:
### 분할기 함수를 사용
texts = text_splitter.split_documents(pages)

print(f"청크(문서)의 갯수 : {len(texts)}")
texts

청크(문서)의 갯수 : 86


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다. 매수 수요 위축으로 주택 매매 거래량이 급감하면서 향후 주택 경기에 대한 부정적 \n시각이 팽배하다. 무엇보다 여전히 높은 금리가 부담으로 작용하고 있다. 주택 경기 불황기에 고금리 \n부담은 주택 수요를 크게 위축시킬 수밖에 없기 때문이다. 다만 주택시장의 주

In [48]:
"""
 * 분할기 클래스와 분할하는 기능을 함수로 만들어서 사용
  - 함수 이름 : getText_Split
  - 반환 받는 값 : 분할이 완료된 결과
  - 출력(print) : 함수를 통해 반환 받은 값을 이용해서 청크(문서)의 갯수 출력하기
"""

loader = PyPDFLoader("./data/2024_KB_부동산_보고서_최종.pdf")
pages = loader.load()
print(f"청크(문서) 갯수 :{len(pages)}")
pages

def getText_Split(p_pages, p_type, p_amount): # 입력받을 값이 몇개든 상관없음.
    ### 의미적 텍스트 분할기 클래스 생성시에는 "분할 방법"과 "분할 기준"을 정의해야함

    rs_text_splitter = SemanticChunker(

        ### 문자를 숫자로 변환하는 임베딩 올라마 모델 정의
        embeddings = OllamaEmbeddings(model="nomic-embed-text"),

        
        breakpoint_threshold_type = p_type,   # 표준편차냐 백분위수냐


        ### 분할 기준 정의 (표준편차 분할에서는 주로 3을 사용)
        breakpoint_threshold_amount = p_amount
    )

    ### 분할기 함수를 사용
    rs_text = rs_text_splitter.split_documents(p_pages)

    return rs_text




청크(문서) 갯수 :84
청크 갯수 :86


In [52]:
### 백분위수
result_text = getText_Split(pages, "percentile", 95)
print(f"청크(문서)의 갯수 : {len(result_text)}")

청크(문서)의 갯수 : 164


In [53]:
### 표준편차
result_text = getText_Split(pages, "standard_deviation", 3)
print(f"청크 갯수 :{len(result_text)}")

청크 갯수 :86


## 분할방법(3) : 사분위수 방식(interquartile)으로 분할하기

In [54]:
### 사분위수 방식
# - 분할방법 : interquartile
# - 분할기준 : 1.5(주로 다른 값 사용해도 됨.)

result_text = getText_Split(pages, "interquartile", 1.5)
print(f"청크 갯수 :{len(result_text)}")
result_text

청크 갯수 :146


[Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 0, 'page_label': '1'}, page_content=''),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title': 'Morning Meeting', 'author': '손은경', 'moddate': '2024-03-04T15:30:01+09:00', 'source': './data/2024_KB_부동산_보고서_최종.pdf', 'total_pages': 84, 'page': 1, 'page_label': '2'}, page_content='1 \n2024 KB 부동산 보고서: 2024년 주택시장 진단과 전망 \n \n \n \nExecutive Summary 1 \n\uf06e 2024년 주택시장 하향 안정 전망, 금리와 공급, 그리고 정책 3대 변수 주목 \n주택경기가 상승과 하락을 반복하면서 주택 경기의 불확실성이 확대되고 있으나. 완만한 하향 조정 \n가능성이 큰 상황이다.'),
 Document(metadata={'producer': 'Microsoft® Word 2016', 'creator': 'Microsoft® Word 2016', 'creationdate': '2024-03-04T15:30:01+09:00', 'title'